# RAG Support Chatbot — Milestone 4 (VS Code / local version)
## MLOps & Monitoring

**Milestone 4 deliverables:**
```
Step 1 → MLflow experiment tracking — log every RAG query automatically
Step 2 → Simulate query load — generate logged runs for the dashboard
Step 3 → Monitoring dashboard — latency, accuracy, volume charts
Step 4 → Retraining pipeline — detect index drift and trigger refresh
Step 5 → View MLflow UI
```

## Step 1 — MLflow experiment tracking

In [1]:
# Run once in venv terminal if not already done:
# pip install mlflow psutil

In [2]:
import importlib
import os
import sys
import time
import json
import random
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from rouge_score import rouge_scorer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Make sure Python can find the src/ package
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.rag_chain as rag_chain
rag_chain = importlib.reload(rag_chain)
load_all      = rag_chain.load_all
ask           = rag_chain.ask
rebuild_index = rag_chain.rebuild_index

load_all()
print('All models loaded.')

c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.
c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\src\rag_chain.py:29: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


[RAG] Configuring Gemini API for embeddings...
[RAG] Gemini embedding model: gemini-embedding-001 (3072-dim)
[RAG] Initialising Groq client for generation...
[RAG] Groq ready: llama-3.3-70b-versatile
[RAG] Loading FAISS index...
[RAG] Index loaded: 800 vectors, dim=3072
[RAG] Loading train lookup table...
[RAG] Loading BM25 corpus...
[RAG] All components loaded. Ready.

All models loaded.


In [ ]:
# ── MLflow setup ──────────────────────────────────────────────────────────────
MLFLOW_EXPERIMENT = "rag-support-chatbot"
MLRUNS_DIR        = os.path.join(PROJECT_ROOT, "mlruns")
DB_PATH           = os.path.join(PROJECT_ROOT, "mlflow.db")

mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")
mlflow.set_experiment(MLFLOW_EXPERIMENT)

rouge  = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

print(f'MLflow tracking URI : sqlite:///{DB_PATH}')
print(f'Experiment name     : {MLFLOW_EXPERIMENT}')
print('MLflow ready.')

MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

In [ ]:
def ask_and_log(
    query: str,
    gold_response: str = None,
    top_k: int = 3,
    use_hybrid: bool = True,
) -> dict:
    """
    Run the full RAG pipeline and log everything to MLflow.

    Logs per run:
        Params  : query, top_k, retrieval_mode, index_size
        Metrics : latency_ms, top1_score, avg_score,
                  bleu (if gold provided), rouge1, rouge2, rougeL
        Artifacts: full result as JSON

    Args:
        query         : Customer question
        gold_response : Ground truth response for BLEU/ROUGE (optional)
        top_k         : Number of docs to retrieve
        use_hybrid    : True = hybrid search, False = vector only

    Returns:
        RAG result dict with added 'latency_ms' key
    """
    with mlflow.start_run():
        # ── Log parameters ────────────────────────────────────────────────────
        mlflow.log_params({
            "query"         : query[:100],  # truncate long queries
            "top_k"         : top_k,
            "retrieval_mode": "hybrid" if use_hybrid else "vector",
            "index_size"    : rag_chain._faiss_index.ntotal,
            "embedding_model": "gemini-embedding-001",
            "llm_model"     : rag_chain.GROQ_MODEL_NAME,
        })

        # ── Run the RAG pipeline ──────────────────────────────────────────────
        start = time.time()
        result = ask(query, top_k=top_k, use_hybrid=use_hybrid)
        latency_ms = (time.time() - start) * 1000
        result["latency_ms"] = latency_ms

        # ── Log retrieval metrics ─────────────────────────────────────────────
        scores = [s["score"] for s in result["sources"]]
        mlflow.log_metrics({
            "latency_ms" : latency_ms,
            "top1_score" : scores[0] if scores else 0.0,
            "avg_score"  : float(np.mean(scores)) if scores else 0.0,
        })

        # ── Log BLEU / ROUGE if gold response provided ────────────────────────
        if gold_response:
            ref = nltk.word_tokenize(gold_response.lower())
            hyp = nltk.word_tokenize(result["answer"].lower())
            bleu = sentence_bleu([ref], hyp, smoothing_function=smooth)

            rs = rouge.score(gold_response, result["answer"])
            mlflow.log_metrics({
                "bleu"   : bleu,
                "rouge1" : rs["rouge1"].fmeasure,
                "rouge2" : rs["rouge2"].fmeasure,
                "rougeL" : rs["rougeL"].fmeasure,
            })
            result["bleu"]   = bleu
            result["rouge1"] = rs["rouge1"].fmeasure

        # ── Log full result as JSON artifact ──────────────────────────────────
        artifact_path = os.path.join(PROJECT_ROOT, "mlruns", "latest_result.json")
        with open(artifact_path, "w") as f:
            json.dump({
                "query"     : result["query"],
                "answer"    : result["answer"],
                "retrieval" : result["retrieval"],
                "latency_ms": latency_ms,
                "sources"   : [
                    {"intent": s["intent"], "score": s["score"]}
                    for s in result["sources"]
                ],
            }, f, indent=2)
        mlflow.log_artifact(artifact_path)

    return result


# --- Quick test ---
print('Testing ask_and_log()...')
result = ask_and_log(
    query="I want to cancel my order",
    gold_response="I can help you cancel your order. Please provide your order number.",
)
print(f'Answer    : {result["answer"][:120]}...')
print(f'Latency   : {result["latency_ms"]:.0f} ms')
print(f'BLEU      : {result.get("bleu", "N/A"):.4f}')
print(f'ROUGE-1   : {result.get("rouge1", "N/A"):.4f}')
print('MLflow run logged successfully.')

## Step 2 — Simulate query load (generate logged runs for the dashboard)

In [ ]:
# Load test set for realistic queries
test_df = pd.read_csv('../data/test_df.csv')
sample  = test_df.sample(30, random_state=42).reset_index(drop=True)

print(f'Running 30 logged queries for monitoring baseline...')
print('This takes ~1-2 minutes (Groq generation per query).\n')

logged_results = []
for i, row in sample.iterrows():
    result = ask_and_log(
        query=row['instruction_clean'],
        gold_response=row['response_clean'],
    )
    logged_results.append(result)
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/30 queries logged...')

print(f'\n30 runs logged to MLflow.')
print(f'Avg latency : {np.mean([r["latency_ms"] for r in logged_results]):.0f} ms')
print(f'Avg BLEU    : {np.mean([r.get("bleu", 0) for r in logged_results]):.4f}')
print(f'Avg ROUGE-1 : {np.mean([r.get("rouge1", 0) for r in logged_results]):.4f}')

## Step 3 — Monitoring dashboard

In [ ]:
def load_mlflow_runs(experiment_name: str = MLFLOW_EXPERIMENT) -> pd.DataFrame:
    """
    Load all MLflow runs for the experiment into a DataFrame.
    Returns one row per run with all logged metrics and params.
    """
    client = mlflow.tracking.MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found.")

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time ASC"]
    )

    records = []
    for run in runs:
        record = {
            "run_id"    : run.info.run_id,
            "start_time": datetime.fromtimestamp(run.info.start_time / 1000),
            **run.data.metrics,
            **{f"param_{k}": v for k, v in run.data.params.items()},
        }
        records.append(record)

    return pd.DataFrame(records)


runs_df = load_mlflow_runs()
print(f'Loaded {len(runs_df)} MLflow runs')
print(f'Columns: {list(runs_df.columns)}')
runs_df.head(3)

In [ ]:
def plot_monitoring_dashboard(runs_df: pd.DataFrame) -> None:
    """
    Generate a 4-panel monitoring dashboard from MLflow run data.

    Panels:
        1. Latency over time (ms per query)
        2. ROUGE-1 score over time (generation quality)
        3. Top-1 retrieval score over time (retrieval quality)
        4. Query volume per hour
    """
    REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
    os.makedirs(REPORTS_DIR, exist_ok=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle('RAG Chatbot — Monitoring Dashboard', fontsize=14, fontweight='bold')

    df = runs_df.sort_values('start_time').reset_index(drop=True)
    x  = range(len(df))

    # Panel 1: Latency over time
    ax = axes[0, 0]
    if 'latency_ms' in df.columns:
        ax.plot(x, df['latency_ms'], color='steelblue', linewidth=1.5, marker='o', markersize=3)
        ax.axhline(df['latency_ms'].mean(), color='red', linestyle='--',
                   label=f"Mean: {df['latency_ms'].mean():.0f} ms")
        ax.set_title('Response Latency (ms)')
        ax.set_xlabel('Query #')
        ax.set_ylabel('Latency (ms)')
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 2: ROUGE-1 over time
    ax = axes[0, 1]
    if 'rouge1' in df.columns:
        ax.plot(x, df['rouge1'], color='teal', linewidth=1.5, marker='o', markersize=3)
        ax.axhline(df['rouge1'].mean(), color='red', linestyle='--',
                   label=f"Mean: {df['rouge1'].mean():.3f}")
        ax.axhline(0.40, color='orange', linestyle=':', label='Target: 0.40')
        ax.set_title('ROUGE-1 Score (Generation Quality)')
        ax.set_xlabel('Query #')
        ax.set_ylabel('ROUGE-1')
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: Top-1 retrieval score over time
    ax = axes[1, 0]
    if 'top1_score' in df.columns:
        ax.plot(x, df['top1_score'], color='purple', linewidth=1.5, marker='o', markersize=3)
        ax.axhline(df['top1_score'].mean(), color='red', linestyle='--',
                   label=f"Mean: {df['top1_score'].mean():.2f}")
        ax.set_title('Top-1 Retrieval Score')
        ax.set_xlabel('Query #')
        ax.set_ylabel('Score')
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 4: Query volume (bar chart)
    ax = axes[1, 1]
    if 'param_retrieval_mode' in df.columns:
        mode_counts = df['param_retrieval_mode'].value_counts()
        ax.bar(mode_counts.index, mode_counts.values,
               color=['steelblue', 'teal'], edgecolor='white')
        ax.set_title('Queries by Retrieval Mode')
        ax.set_ylabel('Count')
    else:
        ax.bar(['Total'], [len(df)], color='steelblue')
        ax.set_title('Total Queries Logged')
        ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    save_path = os.path.join(REPORTS_DIR, 'monitoring_dashboard.png')
    plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Dashboard saved to {save_path}')


plot_monitoring_dashboard(runs_df)

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────
print('='*50)
print('MONITORING SUMMARY')
print('='*50)

metrics = ['latency_ms', 'rouge1', 'rouge2', 'rougeL', 'bleu', 'top1_score']
for m in metrics:
    if m in runs_df.columns:
        col = runs_df[m].dropna()
        print(f'  {m:<15} mean={col.mean():.4f}  min={col.min():.4f}  max={col.max():.4f}')

print(f'\n  Total runs logged : {len(runs_df)}')
print(f'  Index size        : {rag_chain._faiss_index.ntotal:,} vectors')

## Step 4 — Retraining pipeline

In [ ]:
def check_retraining_needed(
    runs_df: pd.DataFrame,
    rouge1_threshold: float = 0.40,
    latency_threshold_ms: float = 5000,
    window: int = 20,
) -> dict:
    """
    Analyse recent MLflow runs to determine if retraining is needed.

    Triggers retraining if any of these are true over the last `window` runs:
        1. Avg ROUGE-1 drops below rouge1_threshold (quality degradation)
        2. Avg latency exceeds latency_threshold_ms (performance degradation)
        3. Index is < 50% complete (fewer vectors than training rows)

    Args:
        runs_df              : DataFrame of MLflow runs from load_mlflow_runs()
        rouge1_threshold     : Min acceptable avg ROUGE-1 score
        latency_threshold_ms : Max acceptable avg latency in ms
        window               : Number of recent runs to analyse

    Returns:
        dict with 'needs_retraining' bool and 'reasons' list
    """
    recent  = runs_df.tail(window)
    reasons = []

    # Check 1: ROUGE-1 quality
    if 'rouge1' in recent.columns:
        avg_rouge1 = recent['rouge1'].dropna().mean()
        if avg_rouge1 < rouge1_threshold:
            reasons.append(
                f"ROUGE-1 ({avg_rouge1:.3f}) below threshold ({rouge1_threshold})"
            )

    # Check 2: Latency
    if 'latency_ms' in recent.columns:
        avg_latency = recent['latency_ms'].dropna().mean()
        if avg_latency > latency_threshold_ms:
            reasons.append(
                f"Avg latency ({avg_latency:.0f} ms) exceeds threshold ({latency_threshold_ms} ms)"
            )

    # Check 3: Index completeness
    index_size  = rag_chain._faiss_index.ntotal
    train_size  = len(rag_chain._train_df)
    completeness = index_size / train_size
    if completeness < 0.5:
        reasons.append(
            f"Index only {completeness*100:.1f}% complete ({index_size:,}/{train_size:,} rows)"
        )

    return {
        "needs_retraining": len(reasons) > 0,
        "reasons"         : reasons,
        "index_completeness": f"{completeness*100:.1f}%",
        "avg_rouge1"      : float(recent['rouge1'].dropna().mean()) if 'rouge1' in recent.columns else None,
        "avg_latency_ms"  : float(recent['latency_ms'].dropna().mean()) if 'latency_ms' in recent.columns else None,
    }


status = check_retraining_needed(runs_df)

print('='*50)
print('RETRAINING STATUS CHECK')
print('='*50)
print(f'  Needs retraining  : {status["needs_retraining"]}')
print(f'  Index completeness: {status["index_completeness"]}')
print(f'  Avg ROUGE-1       : {status["avg_rouge1"]:.4f}' if status['avg_rouge1'] else '  Avg ROUGE-1       : N/A')
print(f'  Avg latency       : {status["avg_latency_ms"]:.0f} ms' if status['avg_latency_ms'] else '  Avg latency       : N/A')

if status['needs_retraining']:
    print('\n  Triggers:')
    for reason in status['reasons']:
        print(f'    - {reason}')
    print('\n  Action: Run rebuild_index() to refresh embeddings.')
else:
    print('\n  System healthy — no retraining needed.')

In [ ]:
def retraining_pipeline(force: bool = False) -> None:
    """
    Full retraining pipeline — checks if needed, then runs rebuild_index().

    In production this would be scheduled (e.g. daily cron job or
    Azure ML pipeline). Here it's triggered manually or by the status check.

    Args:
        force: Skip the status check and rebuild regardless
    """
    print('[Retraining] Checking system status...')
    status = check_retraining_needed(runs_df)

    if not status['needs_retraining'] and not force:
        print('[Retraining] System healthy — skipping rebuild.')
        return

    print('[Retraining] Retraining triggered.')
    if status['reasons']:
        for r in status['reasons']:
            print(f'  Reason: {r}')

    print('[Retraining] Starting rebuild_index()...')

    with mlflow.start_run(run_name="retraining"):
        mlflow.log_params({
            "trigger"   : "auto" if not force else "manual",
            "reasons"   : str(status['reasons']),
            "timestamp" : datetime.now().isoformat(),
        })

        start = time.time()
        rebuild_index()   # runs the full embedding rebuild
        duration_min = (time.time() - start) / 60

        mlflow.log_metrics({
            "rebuild_duration_min": duration_min,
            "new_index_size"      : rag_chain._faiss_index.ntotal,
        })

    print(f'[Retraining] Done. New index: {rag_chain._faiss_index.ntotal:,} vectors.')


# Uncomment to run retraining:
# retraining_pipeline()          # auto — only runs if status check triggers it
# retraining_pipeline(force=True) # manual — always runs
print('Retraining pipeline defined. Uncomment above to run.')

## Step 5 — View MLflow UI

In [ ]:
print('To view the MLflow UI, run this in a NEW terminal (venv activated):')
print()
print(f'  mlflow ui --backend-store-uri sqlite:///{DB_PATH} --port 5000')
print()
print('Then open: http://localhost:5000')
print()
print('You will see:')
print('  - All logged runs with metrics and params')
print('  - Charts for latency, ROUGE, retrieval scores')
print('  - Downloadable artifacts (JSON results)')

## Milestone 4 — Summary

| Deliverable | Status | Where |
|---|---|---|
| Experiment tracking | Done | MLflow — every query auto-logged |
| Monitoring dashboard | Done | `reports/monitoring_dashboard.png` |
| Latency monitoring | Done | Panel 1 of dashboard |
| Quality monitoring (ROUGE) | Done | Panel 2 of dashboard |
| Retrieval monitoring | Done | Panel 3 of dashboard |
| Retraining trigger | Done | `check_retraining_needed()` |
| Retraining pipeline | Done | `retraining_pipeline()` |

---
**Next -> Milestone 5:** Final report + presentation slides.